# MATCH — SQL Graph Database w SQL Server, notatki referencyjne

To jest rzadko używana, ale realna funkcjonalność SQL Server (od wersji 2017): natywne tabele grafowe (node/edge) i klauzula `MATCH` do przeszukiwania wzorców relacji. Przykład zaadaptowany do Twojego kontekstu kadrowego: hierarchia podległości służbowej (kto komu podlega) — klasyczny przypadek, gdzie grafy bywają czytelniejsze niż rekurencyjne CTE.

## 1. Kiedy w ogóle sięgać po graf zamiast zwykłych tabel — kontekst przed składnią

SQL Server od dawna radzi sobie z hierarchiami przez **rekurencyjne CTE** (`WITH RECURSIVE`, a właściwie `WITH ... AS (... UNION ALL ...)` z odwołaniem do samego siebie). Graf (`MATCH`) to **alternatywa**, nie zamiennik uniwersalny — ma sens, gdy:

- Relacji jest **wiele różnych typów** między tymi samymi lub różnymi encjami (np. "podlega służbowo", "mentoruje", "zastępuje podczas nieobecności" — kilka nakładających się sieci relacji między pracownikami).
- Zapytania mają być **czytelne w formie "kto z kim jak połączony"**, a nie tylko "znajdź przodków/potomków w jednej hierarchii" — składnia ASCII-art `MATCH` bywa dużo bardziej zrozumiała niż wielopoziomowe rekurencyjne CTE, szczególnie dla kogoś, kto czyta zapytanie po raz pierwszy.
- Planujesz zapytania typu "znajdź najkrótszą ścieżkę między A i B" — `SHORTEST_PATH` (sekcja 5) to gotowa funkcja, której rekurencyjne CTE nie oferuje wprost.

**Dla prostej, jednopoziomowej hierardlii przełożony→podwładny, którą i tak masz już w `dim_baza_kadrowa` jako kolumnę `ID_Przelozonego`, zwykłe rekurencyjne CTE pozostaje prostsze i wystarczające.** Graf zaczyna się opłacać przy **wielu typach powiązanych relacji naraz** albo faktycznie złożonych, sieciowych strukturach (nie tylko drzewie).

## 2. Tworzenie tabel grafowych — `AS NODE` i `AS EDGE`

```sql
CREATE TABLE dbo.Pracownik (
    ID INT PRIMARY KEY,
    Imie NVARCHAR(100),
    Stanowisko NVARCHAR(100)
) AS NODE;

CREATE TABLE dbo.PodlegaSluzbowo (
    DataOd DATE
) AS EDGE;
```

**`AS NODE`** — dodaje do tabeli ukryte, systemowe kolumny (`$node_id`) identyfikujące wiersz jako węzeł grafu — poza tym zachowuje się jak zwykła tabela (możesz mieć dowolne kolumny, indeksy, ograniczenia).

**`AS EDGE`** — tabela krawędzi, reprezentująca **relację skierowaną** między dwoma węzłami. Automatycznie dostaje kolumny `$from_id`/`$to_id` (wskazujące węzeł początkowy/końcowy) — możesz dodać własne kolumny opisujące samą relację (tu: `DataOd` — od kiedy obowiązuje podległość).

### Wstawianie danych — węzły jak zwykle, krawędzie z jawnym wskazaniem `$node_id`

```sql
INSERT INTO dbo.Pracownik (ID, Imie, Stanowisko) VALUES
    (1, 'Anna Kowalska', 'Dyrektor Regionu'),
    (2, 'Piotr Nowak', 'Kierownik Oddziału'),
    (3, 'Ewa Wiśniewska', 'Specjalista');

INSERT INTO dbo.PodlegaSluzbowo ($from_id, $to_id, DataOd) VALUES
    ((SELECT $node_id FROM dbo.Pracownik WHERE ID = 2), (SELECT $node_id FROM dbo.Pracownik WHERE ID = 1), '2023-01-01'),
    ((SELECT $node_id FROM dbo.Pracownik WHERE ID = 3), (SELECT $node_id FROM dbo.Pracownik WHERE ID = 2), '2024-03-15');
```

Krawędź `PodlegaSluzbowo` skierowana od `$from_id` do `$to_id` — tu: "Piotr podlega Annie", "Ewa podlega Piotrowi". Kierunek ma znaczenie i jest jawny w danych (nie w nazwie kolumny, tylko w samej strukturze `$from_id`/`$to_id`).

## 3. Klauzula `MATCH` — podstawowa składnia ASCII-art

```sql
SELECT Przelozony.Imie, Podwladny.Imie
FROM dbo.Pracownik AS Przelozony, dbo.PodlegaSluzbowo AS Relacja, dbo.Pracownik AS Podwladny
WHERE MATCH (Podwladny-(Relacja)->Przelozony)
  AND Przelozony.Imie = 'Anna Kowalska';
```

**Anatomia wzorca `MATCH`:** `Podwladny-(Relacja)->Przelozony` czyta się dosłownie jak strzałka — "Podwladny, przez krawędź Relacja, wskazuje na Przelozony", zgodnie z kierunkiem `$from_id → $to_id` ustalonym przy wstawianiu danych. To jest odwzorowanie 1:1 tego, jak intuicyjnie rysowałbyś to na kartce (węzły jako kółka, strzałki jako relacje).

**Ważne — tabele w `FROM` wymienione zwykłym przecinkiem (nie `JOIN`), `MATCH` w `WHERE`** — to nietypowa, specyficzna dla grafów składnia; `MATCH` nie jest osobną klauzulą jak `JOIN ... ON`, tylko warunkiem w `WHERE`, który silnik interpretuje jako wzorzec przeszukiwania grafu, nie zwykłe porównanie.

### Wielopoziomowe przeszukiwanie — "przełożony przełożonego"

```sql
SELECT Pracownik1.Imie AS Pracownik, Pracownik3.Imie AS DwaPoziomyWyzej
FROM dbo.Pracownik AS Pracownik1,
     dbo.PodlegaSluzbowo AS Relacja1, dbo.Pracownik AS Pracownik2,
     dbo.PodlegaSluzbowo AS Relacja2, dbo.Pracownik AS Pracownik3
WHERE MATCH (Pracownik1-(Relacja1)->Pracownik2-(Relacja2)->Pracownik3)
  AND Pracownik1.Imie = 'Ewa Wiśniewska';
```

Łańcuch strzałek `Pracownik1-(Relacja1)->Pracownik2-(Relacja2)->Pracownik3` przechodzi przez **dwa** skoki relacji naraz — dla Ewy (podlega Piotrowi, który podlega Annie) zwróci Annę jako "dwa poziomy wyżej". Każdy dodatkowy poziom w hierarchii to kolejny segment strzałki w tym samym wzorcu, bez zagnieżdżania podzapytań.

## 4. Ograniczenia klauzuli `MATCH` — ważne, zanim zaczniesz projektować coś większego

Microsoft wprost dokumentuje twarde ograniczenia tej funkcjonalności:

- **Nie można łączyć wielu wzorców `MATCH` operatorem `OR` ani `NOT`** — tylko `AND` między osobnymi warunkami `MATCH` jest wspierane. To wyklucza np. zapytanie "znajdź pracowników, którzy albo podlegają Annie, albo są mentorowani przez Annię" jako jeden `MATCH` z `OR` — musiałbyś to rozbić na osobne zapytania połączone `UNION`.
- **Ta sama krawędź (alias) nie może wystąpić dwa razy w jednym `MATCH`** — ogranicza pewne bardziej złożone wzorce cykliczne.
- **Kierunek strzałki musi być jawny** — nie da się napisać "relacja w dowolnym kierunku" wprost w jednym wzorcu bez dodatkowej logiki.

To są realne ograniczenia funkcjonalne, nie tylko niedogodności składniowe — jeśli Twój przypadek użycia wymaga logiki `OR`/`NOT` na poziomie wzorca grafu, `MATCH` może się nie nadawać bez sztucznego dzielenia zapytania na części.

## 5. `SHORTEST_PATH` — najkrótsza ścieżka między węzłami (SQL Server 2019+)

To jest funkcjonalność, której **rekurencyjne CTE nie oferuje wprost** (dałoby się to zbudować ręcznie, ale byłoby znacznie bardziej złożone) — bezpośredni argument za sięgnięciem po graf zamiast klasycznej hierarchii.

```sql
SELECT Pracownik1.Imie, STRING_AGG(Posrednicy.Imie, ' -> ') AS Sciezka
FROM dbo.Pracownik AS Pracownik1,
     dbo.PodlegaSluzbowo FOR PATH AS Relacja,
     dbo.Pracownik FOR PATH AS Posrednicy
WHERE MATCH (SHORTEST_PATH(Pracownik1(-(Relacja)->Posrednicy)+))
  AND Pracownik1.Imie = 'Ewa Wiśniewska'
GROUP BY Pracownik1.Imie;
```

`SHORTEST_PATH(...)` wewnątrz `MATCH` z operatorem `+` (jeden lub więcej skoków) przeszukuje graf, znajdując najkrótszą ścieżkę od danego węzła do wszystkich osiągalnych węzłów — przydatne np. do pytania "ile poziomów hierarchii dzieli tego pracownika od zarządu" bez znajomości z góry, ile dokładnie poziomów trzeba przeszukać (w przeciwieństwie do sekcji 3, gdzie liczba segmentów strzałki w `MATCH` odpowiada dokładnej, znanej z góry liczbie poziomów).

## 6. Graf vs rekurencyjne CTE — bezpośrednie porównanie na tym samym zadaniu

**To samo zadanie (wszyscy podwładni Anny, dowolny poziom w dół) — wersja rekurencyjnym CTE, bez grafu, na zwykłej tabeli z kolumną `ID_Przelozonego`:**

```sql
WITH Hierarchia AS (
    SELECT ID, Imie, ID_Przelozonego, 1 AS Poziom
    FROM dim_baza_kadrowa
    WHERE Imie = 'Anna Kowalska'

    UNION ALL

    SELECT p.ID, p.Imie, p.ID_Przelozonego, h.Poziom + 1
    FROM dim_baza_kadrowa p
    JOIN Hierarchia h ON p.ID_Przelozonego = h.ID
)
SELECT * FROM Hierarchia OPTION (MAXRECURSION 100);
```

**Kiedy wybrać które podejście:**

| Kryterium | Rekurencyjne CTE | Graf (`MATCH`) |
|---|---|---|
| Prosta, jedna hierarchia (jedna kolumna typu `ID_Przelozonego`) | ✅ Prostsze, standardowe, każdy programista SQL to zna | Niepotrzebna złożoność |
| Wiele nakładających się typów relacji (podległość + mentoring + zastępstwo) | Wymaga osobnych kolumn/tabel i osobnej logiki na każdą | ✅ Naturalnie modeluje wiele typów krawędzi jednocześnie |
| Najkrótsza ścieżka między dwoma dowolnymi węzłami | Możliwe, ale trzeba budować ręcznie | ✅ `SHORTEST_PATH` gotowe |
| Zespół zna dobrze rekurencyjne CTE, nie zna składni grafowej | ✅ Mniejszy próg wejścia, łatwiejsze utrzymanie przez zespół | Wymaga nauki nowej, niszowej składni |
| Potrzeba `OR`/`NOT` w logice przeszukiwania wzorca | ✅ Zwykły SQL, pełna elastyczność | ❌ Ograniczenie z sekcji 4 |

**Uczciwa rekomendacja: dla typowej hierarchii służbowej w Twoich danych kadrowych (jedna kolumna `ID_Przelozonego`, jeden typ relacji), rekurencyjne CTE pozostaje prostszym, bardziej standardowym wyborem.** Graf SQL Server to wartościowe narzędzie do poznania i mieć "w tylnej kieszeni" na wypadek, gdyby projekt kiedyś wymagał modelowania **kilku nakładających się sieci relacji naraz** (nie tylko jednej hierarchii) — to jest scenariusz, w którym realnie zaczyna wygrywać z tradycyjnym podejściem relacyjnym.

## 7. Podsumowanie

| Potrzebujesz | Rozwiązanie |
|---|---|
| Prosta, jedna hierarchia przełożony→podwładny | Rekurencyjne CTE — prostsze, standardowe |
| Wiele różnych, nakładających się typów relacji między tymi samymi encjami | Tabele grafowe (`AS NODE`/`AS EDGE`) + `MATCH` |
| Najkrótsza ścieżka między dwoma węzłami, nieznana z góry liczba poziomów | `SHORTEST_PATH` w `MATCH` |
| Logika `OR`/`NOT` na poziomie wzorca przeszukiwania | Unikaj czystego `MATCH` — rozbij na `UNION` osobnych zapytań albo wróć do klasycznego SQL |
| Zespół nieobeznany z grafową składnią, priorytet: łatwość utrzymania | Rekurencyjne CTE, nawet jeśli graf byłby elegantszy teoretycznie |